# GKP $X_L$ gate optimization — full sweep over $\Delta$, depth, and gate set

Logical $Z \to -Z$ ($\ket{0_L} \to \ket{1_L}$) optimized on the full grid
$\Delta \in \{0.2, 0.3, 0.4\}$ × `n_gates` $\in [6, 100]$ × every registered gate set, with no
early stopping. Each run is saved to its own file the moment it finishes and logged to a manifest, so
the sweep is resumable after a crash or kernel restart, and every analysis cell below reads from disk
rather than from in-memory results.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import jax

jax.config.update("jax_enable_x64", True)  # before any array is created

import csv
import dataclasses
import gc
import json
import time
import traceback
from datetime import datetime, timezone
from pathlib import Path

import jax.numpy as jnp
import jaxquantum as jqt
import matplotlib.pyplot as plt
import numpy as np

from gkp_optimal_control.gate_optimization import (
    GATE_SETS,
    GateBounds,
    OptimizerConfig,
    make_displacement,
    optimize_gate_sequence,
)
from gkp_optimal_control.gate_results import (
    gate_result_path,
    load_gate_result,
    save_gate_result,
)
from gkp_optimal_control.plotting import (
    plot_photon_number,
    plot_wigner,
    set_plot_style,
)
from gkp_optimal_control.states import gkp_states
from gkp_optimal_control.utils import to_ket

set_plot_style()

Run parameters in one place. The sweep is $3 \times 7 \times 3 = 63$ optimizations at
`N_FOCK = 200` and 64 seeds, so `DEPTHS` and `OPT.n_seeds` are the dials to turn down for a first
pass. Gate sets are read from the `GATE_SETS` registry, and per-gate-set keyword arguments go in
`GATE_KWARGS`.

In [ ]:
N_FOCK = 200          # truncation; must hold the Delta = 0.2 target plus mid-circuit excursions
N_LEAK = 10           # top Fock levels carrying the leakage penalty
MAX_DISP = 4.0        # soft cap on |beta| (ECD) / |alpha| (SNAP)
N_SNAP = 10           # Fock phases optimized per SNAP gate

# Square-lattice GKP.
GKP_ALPHA = jnp.sqrt(jnp.pi / 2)        # primitive displacement, logical Z axis
GKP_BETA = 1j * jnp.sqrt(jnp.pi / 2)    # primitive displacement, logical X axis
GKP_CUTOFF = 10                         # lattice-sum truncation, (2c+1)^2 peaks

# ---- sweep grid ----------------------------------------------------------
DELTAS = [0.2, 0.3, 0.4]
DEPTHS = [6, 10, 24, 32, 48, 72, 100]            # every depth, no early stopping

GATE_SET_NAMES = tuple(GATE_SETS)       # registry order; override explicitly if needed
if len(GATE_SET_NAMES) != 3:
    raise ValueError(
        f"GATE_SETS has {len(GATE_SET_NAMES)} entries {GATE_SET_NAMES}; "
        "set GATE_SET_NAMES to the three to sweep."
    )
print("gate sets:", GATE_SET_NAMES)

# Extra optimize_gate_sequence kwargs per gate set (e.g. frozen parameters for
# the conditional-squeezing set). n_snap is handled separately below.
GATE_KWARGS: dict[str, dict] = {gs: {} for gs in GATE_SET_NAMES}

# Thresholds are reporting-only now: nothing stops when they are met.
TARGET_F = 0.99
STRETCH_F = 0.999

DISP_METHOD = "quadrature"
LOSS_TYPE = "log_infidelity"

BOUNDS = GateBounds(max_disp=MAX_DISP, n_leak=N_LEAK, leakage_weight=1.0)

# One budget for every grid point, so curves are comparable: a point that
# fails has failed at a fixed budget, not fixed effort.
OPT = OptimizerConfig(
    n_seeds=32,
    n_adam_iters=2000,
    peak_lr=0.03,
    polish=True,
    polish_maxiter=1000,
    seed=0,
)

# Warm-start refinement of the best circuit per (gate set, Delta).
RUN_REFINE = True
OPT_POLISH = OptimizerConfig(
    n_adam_iters=2000, peak_lr=0.004, polish=True, polish_maxiter=3000
)

# ---- storage -------------------------------------------------------------
# Truncation and cap are in the folder name so changing them never silently
# reuses results computed under different physics.
DATA_ROOT = Path("../../data/gate_optimization")
RUN_NAME = f"gkp_XL_sweep__nfock{N_FOCK}_maxdisp{MAX_DISP:g}_nsnap{N_SNAP}_cut{GKP_CUTOFF}"
RUN_DIR = DATA_ROOT / RUN_NAME
MANIFEST = RUN_DIR / "manifest.jsonl"
SOURCE_NOTEBOOK = "gkp_XL_full_sweep"
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("run dir:", RUN_DIR.resolve())

## Code states

`gkp_states` builds the finite-energy logicals as a weighted lattice sum of coherent states. Unlike
the binomial words these are $\Delta$-dependent and not exactly orthogonal — the residual overlap
$\braket{0_L|1_L}$ is a finite-energy artifact and grows as $\Delta$ does, which caps the achievable
$X_L$ fidelity independently of circuit depth.

The table below is the truncation sanity check. If the top-`N_LEAK` population of a *target* is not
negligible, `N_FOCK` is too small before optimization even starts.

In [ ]:
STATES: dict[float, tuple] = {}


def gkp_pair(delta: float):
    # Cached (|0_L>, |1_L>) at a given envelope; both normalized Qarrays.
    if delta not in STATES:
        STATES[delta] = gkp_states(N_FOCK, GKP_ALPHA, GKP_BETA, delta, GKP_CUTOFF)
    return STATES[delta]


ns = jnp.arange(N_FOCK)


def state_diagnostics(state):
    probs = jnp.abs(to_ket(state)) ** 2
    return float(jnp.sum(ns * probs)), float(jnp.sum(probs[-N_LEAK:]))


vac = jqt.basis(N_FOCK, 0)

print(f"{'Delta':>6} {'n_bar(0_L)':>11} {'n_bar(1_L)':>11} "
      f"{'|<0_L|1_L>|':>12} {'tail(0_L)':>11}")
print("-" * 56)
for delta in DELTAS:
    g0, g1 = gkp_pair(delta)
    nbar0, tail0 = state_diagnostics(g0)
    nbar1, _ = state_diagnostics(g1)
    ovl = float(jnp.abs(jnp.vdot(to_ket(g0), to_ket(g1))))
    print(f"{delta:>6.2f} {nbar0:>11.2f} {nbar1:>11.2f} {ovl:>12.2e} {tail0:>11.2e}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9.5))
for col, delta in enumerate(DELTAS):
    g0, g1 = gkp_pair(delta)
    plot_wigner(g0, x_bound=6, y_bound=6, ax=axes[0][col],
                title=rf"$\ket{{0_L}}$, $\Delta = {delta}$")
    plot_wigner(g1, x_bound=6, y_bound=6, ax=axes[1][col],
                title=rf"$\ket{{1_L}}$, $\Delta = {delta}$")
fig.tight_layout()
plt.show()

## Task

In [1]:
TASK = "X_L"

def make_tasks(delta: float) -> dict:
    # Task table at a given envelope; states are cached across calls.
    g0, g1 = gkp_pair(delta)
    return {
        "X_L": dict(psi_init=g0, psi_targ=g1,
                    batch_reduction="mean", label=r"$\ket{0_L} \to \ket{1_L}$"),
    }

## Baseline: a single displacement

For the ideal code, $D(\alpha)$ with $\alpha = \sqrt{\pi/2}$ maps $\ket{0_L}$ to $\ket{1_L}$. At finite
$\Delta$ the envelope is not translation-invariant, so it falls short of one. It is scored with the
same objective as the task, so any circuit that fails to beat it is doing no useful work.

In [ ]:
displace = make_displacement(N_FOCK, "expm")
D_alpha = displace(GKP_ALPHA)


def coherent_fidelity(unitary, inputs, targets):
    # F = |mean_k <t_k| U |i_k>|^2, matching batch_reduction="coherent".
    ov = jnp.array([
        jnp.vdot(to_ket(t), unitary @ to_ket(i)) for i, t in zip(inputs, targets)
    ])
    return float(jnp.abs(jnp.mean(ov)) ** 2)


X_BASELINE = {}
print(f"{'Delta':>6} {'F[D(alpha)]':>13} {'1 - F':>11}")
print("-" * 32)
for delta in DELTAS:
    g0, g1 = gkp_pair(delta)
    f_base = coherent_fidelity(D_alpha, [g0], [g1])
    X_BASELINE[delta] = f_base
    print(f"{delta:>6.2f} {f_base:>13.6f} {1 - f_base:>11.3e}")

## Storage and sweep driver

Layout on disk:

```
RUN_DIR/
  config.json                 # physics + optimizer settings; checked on every rerun
  manifest.jsonl              # one line per finished run (scalars + path), append-only
  sweep/delta_0p20/...        # gate_result_path(..., gate_set, depth) per run
  refined/delta_0p20/...
  sweep_summary.npz / .csv    # written by the export cell
```

`run_one` returns immediately if the result file exists, so rerunning the sweep cell resumes where it
stopped. A failed run (e.g. GPU OOM at depth 100) is logged with `status="error"` and does not stop the
sweep; since nothing was saved for it, the next rerun retries it. Full results are never held in
memory across the sweep — only the manifest scalars — because 855 results with 64-seed Adam histories
would not fit comfortably.

In [ ]:
def _jsonable(obj):
    if dataclasses.is_dataclass(obj):
        return {k: _jsonable(v) for k, v in dataclasses.asdict(obj).items()}
    if isinstance(obj, dict):
        return {str(k): _jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_jsonable(v) for v in obj]
    if isinstance(obj, (bool, int, float, str)) or obj is None:
        return obj
    return repr(obj)


CONFIG = _jsonable(dict(
    task=TASK, n_fock=N_FOCK, n_leak=N_LEAK, max_disp=MAX_DISP, n_snap=N_SNAP,
    gkp_alpha=complex(GKP_ALPHA), gkp_beta=complex(GKP_BETA), gkp_cutoff=GKP_CUTOFF,
    disp_method=DISP_METHOD, loss_type=LOSS_TYPE, bounds=BOUNDS, optimizer=OPT,
    optimizer_polish=OPT_POLISH, gate_kwargs=GATE_KWARGS,
))
_cfg_path = RUN_DIR / "config.json"
if _cfg_path.exists():
    _old = json.loads(_cfg_path.read_text())
    if _old != CONFIG:
        diff = sorted(k for k in set(_old) | set(CONFIG) if _old.get(k) != CONFIG.get(k))
        raise RuntimeError(f"{_cfg_path} differs from current settings in {diff}; "
                           "change RUN_NAME or delete the run dir.")
else:
    _cfg_path.write_text(json.dumps(CONFIG, indent=2))


def delta_tag(delta: float) -> str:
    return f"delta_{delta:.2f}".replace(".", "p")


def result_path(stage: str, gate_set: str, delta: float, depth: int) -> Path:
    return Path(gate_result_path(str(RUN_DIR / stage / delta_tag(delta)), gate_set, depth))


def peak_disp(res) -> float:
    mags = res.params.get("disp_magnitudes") if hasattr(res.params, "get") else None
    return float(np.max(np.asarray(mags))) if mags is not None and np.size(mags) else float("nan")


def load_manifest() -> dict:
    # Latest row per (stage, gate_set, delta, depth); a later "ok" supersedes an error.
    rows = {}
    if MANIFEST.exists():
        for line in MANIFEST.read_text().splitlines():
            if line.strip():
                r = json.loads(line)
                rows[(r["stage"], r["gate_set"], r["delta"], r["depth"])] = r
    return rows


def _append_manifest(row: dict) -> None:
    with MANIFEST.open("a") as f:
        f.write(json.dumps(row) + "\n")


def _summary_row(res, *, stage, gate_set, delta, depth, wall_s, path, cached):
    psf = np.asarray(res.per_seed_fidelity, dtype=float)
    return dict(
        stage=stage, gate_set=gate_set, delta=float(delta), depth=int(depth), status="ok",
        fidelity=float(res.fidelity), infidelity=float(1 - res.fidelity),
        leakage=float(res.leakage), n_params=int(np.size(res.flat_params)),
        peak_disp=peak_disp(res), n_seeds=int(psf.size),
        seed_f_min=float(psf.min()), seed_f_median=float(np.median(psf)),
        seed_f_max=float(psf.max()), wall_s=wall_s, cached=cached, path=str(path),
        timestamp=datetime.now(timezone.utc).isoformat(timespec="seconds"),
    )


def run_one(gate_set, delta, depth, *, stage="sweep", optimizer=OPT, params0=None,
            manifest=None):
    # Run (or load) one grid point; save result + manifest row. Returns the row.
    manifest = load_manifest() if manifest is None else manifest
    key = (stage, gate_set, float(delta), int(depth))
    path = result_path(stage, gate_set, delta, depth)

    if path.exists():
        prev = manifest.get(key)
        if prev is not None and prev["status"] == "ok":
            return {**prev, "cached": True}              # nothing to do
        res = load_gate_result(path)                     # file from before the manifest
        row = _summary_row(res, stage=stage, gate_set=gate_set, delta=delta, depth=depth,
                           wall_s=float("nan"), path=path, cached=True)
        _append_manifest(row)
        return row

    spec = make_tasks(delta)[TASK]
    t0 = time.perf_counter()
    try:
        res = optimize_gate_sequence(
            gate_set, depth, spec["psi_init"], spec["psi_targ"],
            n_fock=N_FOCK, n_snap=N_SNAP if gate_set == "snap" else None,
            disp_method=DISP_METHOD, loss_type=LOSS_TYPE,
            batch_reduction=spec["batch_reduction"], bounds=BOUNDS,
            optimizer=optimizer, params0=params0, verbose=False,
            **GATE_KWARGS.get(gate_set, {}),
        )
        path.parent.mkdir(parents=True, exist_ok=True)
        save_gate_result(res, path, initial_state=spec["psi_init"], overwrite=True,
                         source_notebook=SOURCE_NOTEBOOK)
    except Exception as exc:                             # keep the sweep alive
        row = dict(stage=stage, gate_set=gate_set, delta=float(delta), depth=int(depth),
                   status="error", error=f"{type(exc).__name__}: {exc}",
                   wall_s=time.perf_counter() - t0,
                   timestamp=datetime.now(timezone.utc).isoformat(timespec="seconds"))
        _append_manifest(row)
        traceback.print_exc(limit=2)
        return row
    row = _summary_row(res, stage=stage, gate_set=gate_set, delta=delta, depth=depth,
                       wall_s=time.perf_counter() - t0, path=path, cached=False)
    _append_manifest(row)
    del res
    return row


def _fmt(row) -> str:
    if row["status"] != "ok":
        return f"ERROR  {row['error'][:80]}"
    return (f"n_params = {row['n_params']:>5}  1 - F = {row['infidelity']:.3e}  "
            f"leak = {row['leakage']:.1e}  peak|d| = {row['peak_disp']:.2f}  "
            + ("(cached)" if row["cached"] else f"({row['wall_s']:.0f}s)"))

## Main sweep

Outer loop over $\Delta$, then gate set, then depth, with every depth run regardless of whether the
threshold has been met. Each depth has a new parameter shape, so nothing compiled for one depth is
reused by the next; clearing the JAX caches between runs keeps compiled executables from
accumulating in device memory over hundreds of runs. Interrupt at any time and rerun to resume.

In [ ]:
n_total = len(DELTAS) * len(GATE_SET_NAMES) * len(DEPTHS)
manifest = load_manifest()
n_done_before = sum(1 for k, r in manifest.items() if k[0] == "sweep" and r["status"] == "ok")
print(f"{n_total} grid points, {n_done_before} already done\n")

t_sweep = time.perf_counter()
for delta in DELTAS:
    for gate_set in GATE_SET_NAMES:
        print(f"=== Delta = {delta} | {gate_set} | depths {DEPTHS[0]}..{DEPTHS[-1]} ===")
        for depth in DEPTHS:
            row = run_one(gate_set, delta, depth, stage="sweep", manifest=manifest)
            manifest[("sweep", gate_set, float(delta), int(depth))] = row
            print(f"  n_gates = {depth:>3}  {_fmt(row)}")
            if not row.get("cached", True) or row["status"] != "ok":
                jax.clear_caches()
                gc.collect()
        print()

n_err = sum(1 for k, r in manifest.items() if k[0] == "sweep" and r["status"] != "ok")
print(f"sweep finished in {(time.perf_counter() - t_sweep) / 3600:.2f} h; "
      f"{n_err} grid points in error (rerun this cell to retry them)")

## Loading the sweep

Everything from here on reads the manifest, so it runs in a fresh kernel without touching the
optimizer. Full results are loaded from disk only for the circuits that are plotted.

In [ ]:
MANIFEST_ROWS = load_manifest()

STYLE_PRESET = {
    "snap": dict(color="steelblue", marker="o", label="SNAP+D"),
    "ecd": dict(color="crimson", marker="s", label="ECD"),
}
_fallback = [dict(color="darkorange", marker="^"), dict(color="seagreen", marker="D"),
             dict(color="purple", marker="v")]
STYLE = {}
for gs in GATE_SET_NAMES:
    STYLE[gs] = STYLE_PRESET.get(gs) or {**_fallback[len(STYLE) % len(_fallback)], "label": gs}


def rows_for(gate_set, delta, stage="sweep"):
    return {k[3]: r for k, r in MANIFEST_ROWS.items()
            if k[0] == stage and k[1] == gate_set and k[2] == float(delta) and r["status"] == "ok"}


def curve(gate_set, delta, field="infidelity"):
    # Sorted (depths, values, n_params) for one (gate set, Delta).
    rows = rows_for(gate_set, delta)
    depths = np.array(sorted(rows), dtype=int)
    vals = np.array([rows[d][field] for d in depths], dtype=float)
    n_par = np.array([rows[d]["n_params"] for d in depths], dtype=int)
    return depths, vals, n_par


def min_depth(gate_set, delta, target_f=TARGET_F):
    depths, infid, _ = curve(gate_set, delta)
    ok = depths[infid <= 1 - target_f]
    return int(ok[0]) if ok.size else None


def best_depth(gate_set, delta, target_f=TARGET_F):
    # min_depth if the threshold was met, else the highest-F depth.
    d = min_depth(gate_set, delta, target_f)
    if d is None:
        rows = rows_for(gate_set, delta)
        if not rows:
            return None
        d = max(rows, key=lambda k: rows[k]["fidelity"])
    return d


def load_result(gate_set, delta, depth, stage="sweep"):
    return load_gate_result(Path(MANIFEST_ROWS[(stage, gate_set, float(delta), int(depth))]["path"]))


def final_state(res):
    # Single input pair: unwrap a length-1 batch if final_qarray returns one.
    out = res.final_qarray()
    return out[0] if isinstance(out, (list, tuple)) else out


n_ok = sum(1 for k, r in MANIFEST_ROWS.items() if k[0] == "sweep" and r["status"] == "ok")
print(f"{n_ok} / {len(DELTAS) * len(GATE_SET_NAMES) * len(DEPTHS)} sweep points available")

## Depth curves

One panel per $\Delta$, gate sets overlaid. Solid line is $1 - F_\star$, dotted $1 - F_{\star\star}$,
dash-dot the $D(\alpha)$ baseline. Because no depth is skipped, non-monotonicity in these curves is
direct evidence of optimizer failure at fixed budget — a deeper circuit can always represent a
shallower one.

In [ ]:
fig, axes = plt.subplots(1, len(DELTAS), figsize=(5 * len(DELTAS), 4.2), sharey=True)
for ax, delta in zip(np.atleast_1d(axes), DELTAS):
    for gs in GATE_SET_NAMES:
        depths, infid, _ = curve(gs, delta)
        if depths.size:
            ax.plot(depths, infid, linewidth=1.2, markersize=3, **STYLE[gs])
    ax.axhline(1 - TARGET_F, color="black", linestyle="--", alpha=0.6, label=r"$1 - F_\star$")
    ax.axhline(1 - STRETCH_F, color="black", linestyle=":", alpha=0.6)
    ax.axhline(1 - X_BASELINE[delta], color="darkgreen", linestyle="-.", alpha=0.8,
               label=r"$D(\alpha)$")
    ax.set_yscale("log")
    ax.set_xlabel("n_gates")
    ax.set_title(rf"$\Delta = {delta}$")
np.atleast_1d(axes)[0].set_ylabel(r"$1 - F$")
np.atleast_1d(axes)[0].legend(fontsize=8)
fig.tight_layout()
plt.show()

Same data against parameter count, the gate-set-agnostic axis.

In [ ]:
fig, axes = plt.subplots(1, len(DELTAS), figsize=(5 * len(DELTAS), 4.2), sharey=True)
for ax, delta in zip(np.atleast_1d(axes), DELTAS):
    for gs in GATE_SET_NAMES:
        _, infid, n_par = curve(gs, delta)
        if n_par.size:
            ax.plot(n_par, infid, linewidth=1.2, markersize=3, **STYLE[gs])
    ax.axhline(1 - TARGET_F, color="black", linestyle="--", alpha=0.6)
    ax.axhline(1 - STRETCH_F, color="black", linestyle=":", alpha=0.6)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("free parameters")
    ax.set_title(rf"$\Delta = {delta}$")
np.atleast_1d(axes)[0].set_ylabel(r"$1 - F$")
np.atleast_1d(axes)[0].legend(fontsize=8)
fig.tight_layout()
plt.show()

Seed spread across depth: median and best-seed infidelity with the min–max band. A band that widens
with depth means the multi-start is thinning out relative to the landscape and more seeds — not more
depth — would move the curve.

In [ ]:
fig, axes = plt.subplots(len(GATE_SET_NAMES), len(DELTAS),
                         figsize=(5 * len(DELTAS), 3.2 * len(GATE_SET_NAMES)),
                         sharex=True, sharey=True, squeeze=False)
for i, gs in enumerate(GATE_SET_NAMES):
    for j, delta in enumerate(DELTAS):
        ax = axes[i][j]
        depths, f_min, _ = curve(gs, delta, "seed_f_min")
        if not depths.size:
            continue
        _, f_med, _ = curve(gs, delta, "seed_f_median")
        _, f_max, _ = curve(gs, delta, "seed_f_max")
        ax.fill_between(depths, 1 - f_max, 1 - f_min, color=STYLE[gs]["color"], alpha=0.2)
        ax.plot(depths, 1 - f_med, color=STYLE[gs]["color"], linestyle="--", linewidth=1)
        ax.plot(depths, 1 - f_max, color=STYLE[gs]["color"], linewidth=1.3)
        ax.axhline(1 - TARGET_F, color="black", linestyle="--", alpha=0.5)
        ax.set_yscale("log")
        ax.set_title(rf"{STYLE[gs]['label']}, $\Delta = {delta}$", fontsize=10)
for ax in axes[-1]:
    ax.set_xlabel("n_gates")
for row in axes:
    row[0].set_ylabel(r"$1 - F$ (seeds)")
fig.tight_layout()
plt.show()

Minimum depth to threshold against $\Delta$ — the headline resource curve. Missing points did not
reach the threshold anywhere in `DEPTHS`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, (f, name) in zip(axes, [(TARGET_F, r"F_\star"), (STRETCH_F, r"F_{\star\star}")]):
    for gs in GATE_SET_NAMES:
        md_ = [min_depth(gs, d, f) for d in DELTAS]
        xs = [d for d, m in zip(DELTAS, md_) if m is not None]
        ys = [m for m in md_ if m is not None]
        ax.plot(xs, ys, linewidth=1.5, markersize=6, **STYLE[gs])
    ax.set_xlabel(r"$\Delta$")
    ax.set_title(rf"min n_gates for ${name} = {f}$")
axes[0].set_ylabel("n_gates")
axes[0].legend(fontsize=9)
fig.tight_layout()
plt.show()

## Resource table

Per $(\Delta, \text{gate set})$: minimum depth clearing each threshold, and the cost of the best
circuit (the $F_\star$ circuit if one exists, otherwise the highest-fidelity depth, marked `*`).

In [ ]:
header = (
    f"{'Delta':>5} {'set':<14} {'d(F*)':>6} {'d(F**)':>7} {'params':>7} {'1-F':>10} "
    f"{'peak|d|':>8} {'leak':>9} {'seed spread':>20} {'time/run':>9} {'errors':>6}"
)
print(header)
print("-" * len(header))
for delta in DELTAS:
    for gs in GATE_SET_NAMES:
        d = best_depth(gs, delta)
        n_err = sum(1 for k, r in MANIFEST_ROWS.items()
                    if k[:3] == ("sweep", gs, float(delta)) and r["status"] != "ok")
        if d is None:
            print(f"{delta:>5.2f} {gs:<14} {'no results':>6}")
            continue
        r = rows_for(gs, delta)[d]
        met = min_depth(gs, delta) is not None
        d_s = min_depth(gs, delta, STRETCH_F)
        spread = f"{r['seed_f_min']:.4f}..{r['seed_f_max']:.4f}"
        print(
            f"{delta:>5.2f} {gs:<14} {str(d) + ('' if met else '*'):>6} "
            f"{('-' if d_s is None else str(d_s)):>7} {r['n_params']:>7} "
            f"{r['infidelity']:>10.2e} {r['peak_disp']:>8.2f} {r['leakage']:>9.1e} "
            f"{spread:>20} {r['wall_s']:>8.0f}s {n_err:>6}"
        )
print(f"\nF* = {TARGET_F}, F** = {STRETCH_F}.  Baselines 1 - F[D(alpha)]: "
      + ", ".join(f"Delta={d}: {1 - X_BASELINE[d]:.2e}" for d in DELTAS))

## Refining the best circuits

The best circuit per $(\Delta, \text{gate set})$ is warm-started with a long, low-learning-rate run
and an extended L-BFGS-B polish, saved under `refined/`. A point that clears threshold only after
refinement means the sweep budget, not the ansatz, was binding there.

In [ ]:
REFINED_DEPTH: dict[tuple[str, float], int] = {}
if RUN_REFINE:
    manifest = load_manifest()
    for delta in DELTAS:
        for gs in GATE_SET_NAMES:
            d = best_depth(gs, delta)
            if d is None:
                continue
            base = rows_for(gs, delta)[d]
            res0 = load_result(gs, delta, d)
            row = run_one(gs, delta, d, stage="refined", optimizer=OPT_POLISH,
                          params0=jnp.asarray(res0.flat_params), manifest=manifest)
            del res0
            jax.clear_caches()
            gc.collect()
            REFINED_DEPTH[(gs, delta)] = d
            if row["status"] == "ok":
                note = "" if row["fidelity"] >= base["fidelity"] else "  (refinement rejected)"
                print(f"Delta = {delta} | {gs:<14} | n_gates = {d:>3} : 1 - F  "
                      f"{base['infidelity']:.3e} -> {row['infidelity']:.3e}{note}")
            else:
                print(f"Delta = {delta} | {gs:<14} | n_gates = {d:>3} : refinement ERROR")
    MANIFEST_ROWS = load_manifest()


def best_final(gs, delta):
    # (result, stage) for the better of sweep and refined at the best depth.
    d = REFINED_DEPTH.get((gs, delta), best_depth(gs, delta))
    if d is None:
        return None, None
    ref = MANIFEST_ROWS.get(("refined", gs, float(delta), int(d)))
    base = MANIFEST_ROWS[("sweep", gs, float(delta), int(d))]
    if ref is not None and ref["status"] == "ok" and ref["fidelity"] >= base["fidelity"]:
        return load_result(gs, delta, d, "refined"), "refined"
    return load_result(gs, delta, d, "sweep"), "sweep"


BEST = {(gs, delta): best_final(gs, delta) for delta in DELTAS for gs in GATE_SET_NAMES}

## Achieved states

Rows are $\Delta$; the first column is the target $\ket{1_L}$, then the best circuit of each gate set.
For ECD-type sets `final_qarray()` returns the $\ket{g}$ block renormalized.

In [ ]:
ncol = 1 + len(GATE_SET_NAMES)
fig, axes = plt.subplots(len(DELTAS), ncol, figsize=(5 * ncol, 4.7 * len(DELTAS)), squeeze=False)
for row, delta in enumerate(DELTAS):
    _, g1 = gkp_pair(delta)
    plot_wigner(g1, x_bound=6, y_bound=6, ax=axes[row][0],
                title=rf"target $\ket{{1_L}}$, $\Delta = {delta}$")
    for col, gs in enumerate(GATE_SET_NAMES, start=1):
        res, _ = BEST[(gs, delta)]
        if res is None:
            axes[row][col].axis("off")
            continue
        plot_wigner(final_state(res), x_bound=6, y_bound=6, ax=axes[row][col],
                    title=rf"{STYLE[gs]['label']}, $N={res.n_gates}$, $1-F = {1 - res.fidelity:.1e}$")
fig.tight_layout()
plt.show()

Photon-number distributions — the truncation check. Weight approaching $n = $ `N_FOCK` means
`MAX_DISP` and `N_FOCK` are mismatched and the reported fidelity is suspect.

In [ ]:
fig, axes = plt.subplots(len(DELTAS), ncol, figsize=(5 * ncol, 3.5 * len(DELTAS)), squeeze=False)
for row, delta in enumerate(DELTAS):
    _, g1 = gkp_pair(delta)
    plot_photon_number(g1, ax=axes[row][0], max_n=60,
                       title=rf"target $\ket{{1_L}}$, $\Delta = {delta}$")
    for col, gs in enumerate(GATE_SET_NAMES, start=1):
        res, _ = BEST[(gs, delta)]
        if res is None:
            axes[row][col].axis("off")
            continue
        plot_photon_number(final_state(res), ax=axes[row][col], max_n=60,
                           title=f"{STYLE[gs]['label']} achieved")
fig.tight_layout()
plt.show()

## Stabilizer expectation values

$\braket{S_q} = \mathrm{Re}\braket{D(2\alpha)}$, $\braket{S_p} = \mathrm{Re}\braket{D(2\beta)}$, and
$\Delta_{\text{eff}} = \sqrt{\ln(1/|\braket{S}|^2)/2\pi}$ with
$|\braket{S}| = \tfrac12(|\braket{S_q}| + |\braket{S_p}|)$. The normalization of
$\Delta_{\text{eff}}$ is convention-dependent, so each achieved value is read against its target's
$\Delta_{\text{eff}}$ rather than against $\Delta$ directly.

In [ ]:
D_2alpha = displace(2 * GKP_ALPHA)
D_2beta = displace(2 * GKP_BETA)


def stabilizers(state):
    # (<S_q>, <S_p>, Delta_eff) for a single cavity ket.
    psi = to_ket(state)
    s_q = float(jnp.real(jnp.vdot(psi, D_2alpha @ psi)))
    s_p = float(jnp.real(jnp.vdot(psi, D_2beta @ psi)))
    s_bar = 0.5 * (abs(s_q) + abs(s_p))
    delta_eff = float(np.sqrt(np.log(1.0 / max(s_bar, 1e-12) ** 2) / (2 * np.pi)))
    return s_q, s_p, delta_eff


STAB = {}
print(f"{'state':<30} {'<S_q>':>8} {'<S_p>':>8} {'Delta_eff':>10}")
print("-" * 60)
for delta in DELTAS:
    _, g1 = gkp_pair(delta)
    STAB[("target", delta)] = stabilizers(g1)
    sq, sp, de = STAB[("target", delta)]
    print(f"{f'target 1_L, Delta={delta}':<30}{sq:>8.4f}{sp:>8.4f}{de:>10.4f}")
    for gs in GATE_SET_NAMES:
        res, _ = BEST[(gs, delta)]
        if res is None:
            continue
        STAB[(gs, delta)] = stabilizers(final_state(res))
        sq, sp, de = STAB[(gs, delta)]
        label = f"  {STYLE[gs]['label']} N={res.n_gates}"
        print(f"{label:<30}{sq:>8.4f}{sp:>8.4f}{de:>10.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
deltas_arr = np.array(DELTAS)
ax.plot(deltas_arr, [STAB[("target", d)][2] for d in DELTAS], color="black", linestyle="--",
        marker="x", alpha=0.7, label=r"target $\Delta_{\rm eff}$")
for gs in GATE_SET_NAMES:
    xs = [d for d in DELTAS if (gs, d) in STAB]
    ax.plot(xs, [STAB[(gs, d)][2] for d in xs], linewidth=1.5, markersize=6, **STYLE[gs])
ax.set_xlabel(r"target $\Delta$")
ax.set_ylabel(r"achieved $\Delta_{\rm eff}$")
ax.set_title(r"Effective squeezing of the $X_L$ output")
ax.legend()
fig.tight_layout()
plt.show()

## Gain over the displacement baseline

In [ ]:
for delta in DELTAS:
    f_base = X_BASELINE[delta]
    print(f"Delta = {delta}: baseline D(alpha)  1 - F = {1 - f_base:.3e}")
    for gs in GATE_SET_NAMES:
        res, stage = BEST[(gs, delta)]
        if res is None:
            continue
        gain = (1 - f_base) / max(1 - res.fidelity, 1e-15)
        print(f"   {STYLE[gs]['label']:<12} N = {res.n_gates:<4} 1 - F = {1 - res.fidelity:.3e}"
              f"   gain = {gain:.1f}x   ({stage})")

## Convergence diagnostics

Adam traces over seeds and the final per-seed infidelity for each best circuit (sweep stage, since
the refined run is single-start). The plotted loss includes the penalties.

In [ ]:
panels = [(gs, d) for d in DELTAS for gs in GATE_SET_NAMES if best_depth(gs, d) is not None]
fig, axes = plt.subplots(len(panels), 2, figsize=(13, 2.8 * len(panels)), squeeze=False)
for row, (gs, delta) in enumerate(panels):
    d = best_depth(gs, delta)
    res = load_result(gs, delta, d, "sweep")
    hist = np.asarray(res.adam_history)
    ax_h, ax_s = axes[row]
    for k in range(hist.shape[0]):
        ax_h.plot(hist[k], color="gray", alpha=0.25, linewidth=0.6)
    ax_h.plot(hist[res.best_seed], color=STYLE[gs]["color"], linewidth=1.4)
    ax_h.set_ylabel("loss")
    ax_h.set_title(rf"{STYLE[gs]['label']} / $\Delta = {delta}$ / N = {d}", fontsize=10)
    sorted_infid = 1 - np.sort(np.asarray(res.per_seed_fidelity))[::-1]
    ax_s.bar(np.arange(sorted_infid.size), sorted_infid, color=STYLE[gs]["color"],
             edgecolor="black", linewidth=0.3)
    ax_s.set_yscale("log")
    ax_s.set_ylabel(r"$1 - F$")
    del res
axes[-1][0].set_xlabel("Adam iteration")
axes[-1][1].set_xlabel("seed (sorted)")
fig.tight_layout()
plt.show()

## Displacement amplitudes

Peak $|\beta|$ / $|\alpha|$ across the whole depth sweep, then per layer for the best circuits. Points
sitting on the dashed cap mean the constraint is binding and the fidelity there is a constrained
optimum. Gate sets that do not report `disp_magnitudes` are skipped.

In [ ]:
fig, axes = plt.subplots(1, len(DELTAS), figsize=(5 * len(DELTAS), 4), sharey=True)
for ax, delta in zip(np.atleast_1d(axes), DELTAS):
    for gs in GATE_SET_NAMES:
        depths, pk, _ = curve(gs, delta, "peak_disp")
        if depths.size and np.isfinite(pk).any():
            ax.plot(depths, pk, linewidth=1.2, markersize=3, **STYLE[gs])
    ax.axhline(MAX_DISP, color="black", linestyle="--", alpha=0.6)
    ax.set_xlabel("n_gates")
    ax.set_title(rf"peak displacement, $\Delta = {delta}$")
np.atleast_1d(axes)[0].set_ylabel(r"max$_i$ $|\beta_i|$ or $|\alpha_i|$")
np.atleast_1d(axes)[0].legend(fontsize=8)
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(len(DELTAS), 1, figsize=(14, 3.2 * len(DELTAS)), squeeze=False)
width = 0.8 / len(GATE_SET_NAMES)
for ax, delta in zip(axes[:, 0], DELTAS):
    for k, gs in enumerate(GATE_SET_NAMES):
        res, _ = BEST[(gs, delta)]
        mags = None if res is None else res.params.get("disp_magnitudes")
        if mags is None:
            continue
        mags = np.asarray(mags)
        idx = np.arange(mags.size) + (k - (len(GATE_SET_NAMES) - 1) / 2) * width
        ax.bar(idx, mags, width=width, color=STYLE[gs]["color"], edgecolor="black",
               linewidth=0.3, label=f"{STYLE[gs]['label']} (N={res.n_gates})")
    ax.axhline(MAX_DISP, color="black", linestyle="--", alpha=0.6)
    ax.set_ylabel(rf"$\Delta = {delta}$")
    ax.legend(fontsize=8)
axes[-1][0].set_xlabel("layer")
fig.tight_layout()
plt.show()

## Optimal parameters

Labelled parameters of the best circuit per $(\Delta, \text{gate set})$. Every other grid point's
parameters are in its own result file (path in the manifest).

In [ ]:
for delta in DELTAS:
    for gs in GATE_SET_NAMES:
        res, stage = BEST[(gs, delta)]
        if res is None:
            continue
        print("=" * 78)
        print(f"Delta = {delta} | {gs.upper()} | n_gates = {res.n_gates} | "
              f"1 - F = {1 - res.fidelity:.3e} | {stage}")
        for name, val in res.params.items():
            arr = np.asarray(val)
            if name in ("per_pair_fidelity", "disp_magnitudes"):
                continue
            if np.iscomplexobj(arr):
                print(f"{name:<14}:", [f"{v:+.4f}" for v in arr.ravel()])
            else:
                print(f"{name:<14}:", np.round(arr, 4))

## Export

The whole sweep as dense arrays indexed `[gate_set, delta, depth]` (NaN where a point is missing or
errored), plus the manifest as CSV for pandas. Flat parameter vectors are not duplicated here — each
lives in its result file, and `flat_params` from any of them is a valid `params0` for a warm start.

In [ ]:
FIELDS = ["fidelity", "infidelity", "leakage", "n_params", "peak_disp",
          "seed_f_min", "seed_f_median", "seed_f_max", "wall_s"]
grid = {f: np.full((len(GATE_SET_NAMES), len(DELTAS), len(DEPTHS)), np.nan) for f in FIELDS}
for i, gs in enumerate(GATE_SET_NAMES):
    for j, delta in enumerate(DELTAS):
        rows = rows_for(gs, delta)
        for k, depth in enumerate(DEPTHS):
            if depth in rows:
                for f in FIELDS:
                    v = rows[depth].get(f)
                    grid[f][i, j, k] = np.nan if v is None else v

export = dict(
    gate_sets=np.array(GATE_SET_NAMES), deltas=np.array(DELTAS), depths=np.array(DEPTHS),
    x_baseline=np.array([X_BASELINE[d] for d in DELTAS]),
    config_json=json.dumps(CONFIG),
    **{f"sweep__{f}": a for f, a in grid.items()},
)
for (gs, delta), (res, stage) in BEST.items():
    if res is None:
        continue
    tag = f"best__{gs}__{delta_tag(delta)}"
    export[f"{tag}__flat_params"] = np.asarray(res.flat_params)
    export[f"{tag}__n_gates"] = res.n_gates
    export[f"{tag}__fidelity"] = res.fidelity
    export[f"{tag}__stage"] = stage

npz_path = RUN_DIR / "sweep_summary.npz"
np.savez(npz_path, **export)

csv_path = RUN_DIR / "sweep_summary.csv"
rows = sorted(MANIFEST_ROWS.values(), key=lambda r: (r["stage"], r["gate_set"], r["delta"], r["depth"]))
keys = sorted({k for r in rows for k in r})
with csv_path.open("w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=keys)
    w.writeheader()
    w.writerows(rows)
print(f"wrote {npz_path} ({len(export)} arrays) and {csv_path} ({len(rows)} rows)")

## Notes

**Findings.**

- Minimum depth at $F_\star$ per gate set, $\Delta = 0.2 / 0.3 / 0.4$: _
- Same at $F_{\star\star}$: _
- Cheapest gate set by parameter count at each $\Delta$: _
- Non-monotone stretches of the depth curves (budget-limited depths): _
- Was `MAX_DISP` binding, and from which depth on? _
- Gain over $D(\alpha)$ per $\Delta$: _

**Open items.**

- Depth-annealed warm starts (pad depth $N$'s solution with identity layers to seed $N+1$) would make
  the curves monotone by construction; it needs the `flat_params` layout per gate set.
- Add the $\ket{1_L} \to \ket{0_L}$ pair to certify the full logical $X_L$ rather than one action.
- Sweep `N_SNAP` and `MAX_DISP` (with `N_FOCK` raised to match); both are fixed here.
- Check GPU memory at depth 100 × 64 seeds × `N_FOCK = 200`; OOM points show up as `status="error"`
  in the manifest and can be rerun with fewer seeds under a separate `RUN_NAME`.